In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel,PeftConfig
import torch
import safetensors



In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights: 100%|██████████| 434/434 [00:13<00:00, 31.46it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


In [7]:
# ✅ 1. Paths
SAVE_DIR="Final_all/outputs"
base_model_path = "Qwen/Qwen2.5-3B-Instruct"
lora_adapter_path = f"/home/souradeep/Desktop/Customer Agentic System/Final_all/qwen_lora_adapter"

# ✅ 2. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# ✅ 3. Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

# ✅ 4. Load LoRA adapter configuration
peft_config = PeftConfig.from_pretrained(lora_adapter_path)

# ✅ 5. Create a new PeftModel instance
model = PeftModel(base_model, peft_config)

# ✅ 6. Load the LoRA adapter weights using safetensors
adapter_weights = safetensors.torch.load_file(f"{lora_adapter_path}/model.safetensors")
model.load_state_dict(adapter_weights, strict=False)

print("✅ LoRA loaded successfully")

# ✅ 7. Move the model to GPU
model.to("cuda")

print("✅ Model loaded on GPU")

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 721.44it/s]


✅ LoRA loaded successfully
✅ Model loaded on GPU


In [13]:
query = "Can u help me with my billing issue?"

SYSTEM_PROMPT = """You are a customer support query classifier.
Classify the user query into exactly one of these categories:
[billing, account, technical, shipping, order, complaint, other]
Respond with ONLY the category name, nothing else."""

input_text = f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n<|im_start|>user\n{query}\n<|im_end|>\n<|im_start|>assistant\n"

# 1. Call tokenizer() directly instead of tokenizer.encode()
# This returns a dictionary containing both 'input_ids' and 'attention_mask'
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    # 2. Pass both explicitly from the inputs dictionary
    output = model.generate(
        input_ids=inputs["input_ids"], 
        attention_mask=inputs["attention_mask"],
        max_new_tokens=5
    )

generated_text = tokenizer.decode(output[0], skip_special_tokens=True).strip().lower()
predicted_category = generated_text.split("assistant")[-1].strip()

print(f"Category: {predicted_category}")

Category: billing
